# 01 Build Dataset FINAL v2

This notebook builds the multivariate container anomaly-detection dataset directly from the raw Alibaba archives. It extracts and parses the `tar.gz` files, joins container usage with container metadata, joins machine usage and machine metadata where required, derives contextual features from metadata, and saves sliding-window artifacts for the downstream FiLM autoencoder pipeline.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from container_ad_pipeline.config import PipelineConfig, ensure_pipeline_directories
from container_ad_pipeline.dataset import (
    assign_splits,
    generate_alibaba_windows,
    join_alibaba_frames,
    load_alibaba_raw_frames,
    preprocess_joined_alibaba_frame,
    save_dataset_bundle,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

config = PipelineConfig()
ensure_pipeline_directories(config)

# Notebook-friendly limits for end-to-end execution. Increase these for larger experiments.
config.dataset.max_container_meta_files = 10
config.dataset.max_container_usage_files = 10
config.dataset.max_machine_meta_files = None
config.dataset.max_machine_usage_files = 10
config.dataset.max_usage_rows = 20_000_000
config.dataset.max_machine_usage_rows = 20_000_000
config.dataset.max_containers = 1000
config.dataset.chunksize = 5_000

archive_summary = pd.Series(
    {
        "raw_container_meta_tar": str(config.paths.raw_container_meta_tar),
        "raw_container_usage_tar": str(config.paths.raw_container_usage_tar),
        "raw_machine_meta_tar": str(config.paths.raw_machine_meta_tar),
        "raw_machine_usage_tar": str(config.paths.raw_machine_usage_tar),
        "processed_dir": str(config.paths.processed_dir),
        "window_size": config.dataset.window_size,
        "stride": config.dataset.stride,
        "feature_columns": config.dataset.feature_columns,
        "context_numeric_columns": config.dataset.context_numeric_columns,
        "context_categorical_columns": config.dataset.context_categorical_columns,
        "max_usage_rows": config.dataset.max_usage_rows,
        "max_machine_usage_rows": config.dataset.max_machine_usage_rows,
        "max_containers": config.dataset.max_containers,
    },
    name="value",
)
display(archive_summary)


raw_container_meta_tar         C:\Users\kaspe\Desktop\Project\dataset\data\co...
raw_container_usage_tar        C:\Users\kaspe\Desktop\Project\dataset\data\co...
raw_machine_meta_tar           C:\Users\kaspe\Desktop\Project\dataset\data\ma...
raw_machine_usage_tar          C:\Users\kaspe\Desktop\Project\dataset\data\ma...
processed_dir                  C:\Users\kaspe\Desktop\Project\project\data\re...
window_size                                                                   24
stride                                                                         6
feature_columns                [cpu_util, mem_util, cpi, mem_gps, mpki, net_i...
context_numeric_columns        [container_cpu_request, container_cpu_limit, c...
context_categorical_columns    [container_app_du, container_status, machine_f...
max_usage_rows                                                          20000000
max_machine_usage_rows                                                  20000000
max_containers              

## Load Raw Alibaba Archives

The archive reader streams the `tar.gz` members directly, so the dataset can be built from the original Alibaba files without first materializing a separate CSV. The inspection below confirms the raw frames available for container usage, container metadata, machine usage, and machine metadata.


In [2]:
frames = load_alibaba_raw_frames(config.paths, config.dataset)
frame_shapes = pd.Series({name: frame.shape for name, frame in frames.items()}, name="shape")
display(frame_shapes)

display(frames["container_usage"].head())
display(frames["container_meta"].head())
display(frames["machine_usage"].head())
display(frames["machine_meta"].head())


container_meta        (5441, 8)
container_usage    (999400, 11)
machine_meta          (3218, 7)
machine_usage      (4117703, 9)
Name: shape, dtype: object

,container_id,machine_id,time_stamp,cpu_util,mem_util,cpi,mem_gps,mpki,net_in,net_out,disk_io
0,c_13781,m_944,86400.0,11.0,99.0,NaN,NaN,NaN,0.78,1.01,5.0
1,c_10950,m_3000,86400.0,1.0,27.0,NaN,NaN,NaN,0.14,0.13,NaN
2,c_28492,m_3017,86400.0,1.0,16.0,NaN,NaN,NaN,0.01,0.01,3.0
3,c_20267,m_1740,86400.0,2.0,58.0,NaN,NaN,NaN,0.00,0.00,5.0
4,c_20622,m_2063,86400.0,0.0,66.0,NaN,NaN,NaN,0.00,0.00,4.0


,container_id,machine_id,time_stamp,app_du,status,cpu_request,cpu_limit,mem_size
0,c_17985,m_369,0.0,app_3050,started,400.0,480.0,1.56
1,c_103,m_590,0.0,app_9059,started,400.0,480.0,1.95
2,c_18032,m_598,0.0,app_82,started,400.0,400.0,1.56
3,c_115,m_131,0.0,app_7414,started,800.0,800.0,3.13
4,c_119,m_265,0.0,app_2791,started,400.0,400.0,1.56


,machine_id,time_stamp,machine_cpu_util,machine_mem_util,machine_mem_gps,machine_mpki,machine_net_in,machine_net_out,machine_disk_io
0,m_635,0.0,14.0,89.0,NaN,NaN,38.08,30.11,3.0
1,m_658,0.0,12.0,92.0,NaN,NaN,40.19,29.44,3.0
2,m_2620,0.0,14.0,91.0,NaN,NaN,24.70,22.87,3.0
3,m_3892,0.0,29.0,83.0,NaN,NaN,41.72,33.34,57.0
4,m_129,0.0,14.0,89.0,NaN,NaN,38.47,32.76,5.0


,machine_id,time_stamp,failure_domain_1,failure_domain_2,cpu_num,mem_size,status
0,m_10,0.0,128,17,96.0,100.0,USING
1,m_994,0.0,68,17,96.0,100.0,USING
2,m_993,0.0,80,1,96.0,100.0,USING
3,m_992,0.0,55,3,96.0,100.0,USING
4,m_2452,0.0,45,19,96.0,100.0,USING


## Join Container And Machine Metadata

Container usage is aligned with the most recent available container metadata for the same `container_id`. Machine usage and machine metadata are then aligned by `machine_id` using backward `asof` joins, which preserves the temporal order of the raw records while adding contextual state to each usage observation.


In [3]:
joined = join_alibaba_frames(frames, config.dataset)
print(joined.shape)
join_columns = [
    "container_id",
    "machine_id",
    "time_stamp",
    *config.dataset.feature_columns,
    "container_app_du",
    "container_status",
    "container_cpu_request",
    "container_cpu_limit",
    "container_mem_size",
    "machine_status",
    "machine_cpu_num",
    "machine_mem_size",
]
display(joined[[column for column in join_columns if column in joined.columns]].head(10))


(999400, 32)


,container_id,machine_id,time_stamp,cpu_util,mem_util,cpi,mem_gps,mpki,net_in,net_out,disk_io,container_app_du,container_status,container_cpu_request,container_cpu_limit,container_mem_size,machine_status,machine_cpu_num,machine_mem_size
0,c_24154,m_10,86410.0,27.0,100.0,NaN,NaN,NaN,0.49,0.45,4.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
1,c_24154,m_10,86870.0,19.0,99.0,NaN,NaN,NaN,0.49,0.45,4.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
2,c_24154,m_10,86890.0,22.0,99.0,NaN,NaN,NaN,0.49,0.45,6.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
3,c_24154,m_10,87560.0,17.0,99.0,NaN,NaN,NaN,0.49,0.45,4.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
4,c_24154,m_10,87650.0,17.0,99.0,NaN,NaN,NaN,0.49,0.45,5.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
5,c_24154,m_10,88290.0,16.0,100.0,NaN,NaN,NaN,0.49,0.45,4.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
6,c_24154,m_10,88560.0,13.0,98.0,NaN,NaN,NaN,0.49,0.45,4.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
7,c_24154,m_10,90120.0,14.0,99.0,NaN,NaN,NaN,0.49,0.45,8.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
8,c_24154,m_10,90880.0,14.0,99.0,NaN,NaN,NaN,0.49,0.45,5.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0
9,c_24154,m_10,91040.0,13.0,99.0,NaN,NaN,NaN,0.49,0.45,6.0,app_7223,started,400.0,400.0,1.56,USING,96.0,100.0


## Preprocess Features And Derive Context

The multivariate usage features are cleaned groupwise per container, clipped to non-negative values, and transformed with `log1p`. Numeric context is derived from the metadata columns, while categorical metadata is ordinal-encoded so the FiLM autoencoder can condition on a compact context vector.


In [4]:
preprocessed, context_encoder = preprocess_joined_alibaba_frame(joined, config.dataset)
summary_df = preprocessed[config.dataset.feature_columns].describe().T
summary_df["missing_fraction"] = preprocessed[config.dataset.feature_columns].isna().mean()
display(summary_df)

display(preprocessed[[
    "container_id",
    "machine_id",
    "time_stamp",
    *config.dataset.context_columns,
]].head(10))

context_encoder


,count,mean,std,min,25%,50%,75%,max,missing_fraction
cpu_util,999400.0,1.903295,0.981826,0.0,1.098612,1.945910,2.639057,4.615120,0.0
mem_util,999400.0,4.326279,0.470336,0.0,4.234107,4.553877,4.605170,4.615120,0.0
cpi,999400.0,0.874426,0.241060,0.0,0.832909,0.912283,0.989541,2.551006,0.0
mem_gps,999400.0,0.044233,0.126591,0.0,0.000000,0.009950,0.029559,3.029650,0.0
mpki,999400.0,0.368131,0.362979,0.0,0.000000,0.693147,0.693147,2.833213,0.0
net_in,999400.0,0.133654,0.210117,0.0,0.009950,0.048790,0.165514,2.022871,0.0
net_out,999400.0,0.137916,0.216998,0.0,0.009950,0.048790,0.173953,2.196113,0.0
disk_io,999400.0,2.105639,0.587970,0.0,1.791759,2.079442,2.397895,4.624973,0.0


,container_id,machine_id,time_stamp,container_cpu_request,container_cpu_limit,container_mem_size,machine_cpu_num,machine_mem_size,machine_cpu_util,machine_mem_util,machine_mem_gps,machine_mpki,machine_net_in,machine_net_out,machine_disk_io,container_app_du,container_status,machine_failure_domain_1,machine_failure_domain_2,machine_status
0,c_24154,m_10,86410.0,400.0,400.0,1.56,96.0,100.0,20.0,91.0,0.0,0.0,35.790001,28.750000,4.0,525.0,0.0,31.0,8.0,0.0
1,c_24154,m_10,86870.0,400.0,400.0,1.56,96.0,100.0,37.0,95.0,0.0,0.0,35.799999,28.760000,3.0,525.0,0.0,31.0,8.0,0.0
2,c_24154,m_10,86890.0,400.0,400.0,1.56,96.0,100.0,54.0,96.0,0.0,0.0,35.799999,28.760000,6.0,525.0,0.0,31.0,8.0,0.0
3,c_24154,m_10,87560.0,400.0,400.0,1.56,96.0,100.0,29.0,88.0,0.0,0.0,35.799999,28.760000,3.0,525.0,0.0,31.0,8.0,0.0
4,c_24154,m_10,87650.0,400.0,400.0,1.56,96.0,100.0,45.0,87.0,0.0,0.0,35.810001,28.770000,5.0,525.0,0.0,31.0,8.0,0.0
5,c_24154,m_10,88290.0,400.0,400.0,1.56,96.0,100.0,37.0,88.0,0.0,0.0,35.820000,28.780001,4.0,525.0,0.0,31.0,8.0,0.0
6,c_24154,m_10,88560.0,400.0,400.0,1.56,96.0,100.0,26.0,83.0,0.0,0.0,35.820000,28.780001,5.0,525.0,0.0,31.0,8.0,0.0
7,c_24154,m_10,90120.0,400.0,400.0,1.56,96.0,100.0,38.0,88.0,0.0,0.0,35.840000,28.790001,5.0,525.0,0.0,31.0,8.0,0.0
8,c_24154,m_10,90880.0,400.0,400.0,1.56,96.0,100.0,29.0,87.0,0.0,0.0,35.849998,28.799999,4.0,525.0,0.0,31.0,8.0,0.0
9,c_24154,m_10,91040.0,400.0,400.0,1.56,96.0,100.0,37.0,89.0,0.0,0.0,35.849998,28.799999,4.0,525.0,0.0,31.0,8.0,0.0


{'categorical_columns': ['container_app_du',
  'container_status',
  'machine_failure_domain_1',
  'machine_failure_domain_2',
  'machine_status'],
 'numeric_columns': ['container_cpu_request',
  'container_cpu_limit',
  'container_mem_size',
  'machine_cpu_num',
  'machine_mem_size',
  'machine_cpu_util',
  'machine_mem_util',
  'machine_mem_gps',
  'machine_mpki',
  'machine_net_in',
  'machine_net_out',
  'machine_disk_io'],
 'categories': {'container_app_du': ['app_10',
   'app_1008',
   'app_1012',
   'app_1027',
   'app_1059',
   'app_1063',
   'app_1093',
   'app_1103',
   'app_1118',
   'app_1123',
   'app_1128',
   'app_1133',
   'app_114',
   'app_1140',
   'app_119',
   'app_1201',
   'app_1215',
   'app_1225',
   'app_1233',
   'app_1244',
   'app_1254',
   'app_1276',
   'app_1293',
   'app_1295',
   'app_130',
   'app_1308',
   'app_1347',
   'app_135',
   'app_1379',
   'app_1398',
   'app_141',
   'app_1414',
   'app_1421',
   'app_1423',
   'app_1429',
   'app_1473',
 

## Create Sliding Windows And Dataset Splits

Sliding windows are generated from the raw joined usage records on a per-container basis. Each window keeps the multivariate sequence `X`, the aligned context vector `C`, and metadata describing the container, machine, and timestamps for later interpretation and GPT adjudication.


In [5]:
bundle = generate_alibaba_windows(preprocessed, config.dataset, context_encoder)
bundle.metadata = assign_splits(bundle.metadata, config.dataset)

print("X shape:", bundle.X.shape)
print("C shape:", bundle.C.shape)
display(bundle.metadata.head(10))
display(bundle.metadata["split"].value_counts().rename_axis("split").to_frame("windows"))
display(pd.Series(bundle.dataset_meta, name="dataset_meta"))


X shape: (163140, 24, 8)
C shape: (163140, 17)


,window_id,entity_id,container_id,machine_id,start_index,end_index,start_time,end_time,app_du,container_status,machine_status,split
0,3874,c_18245,c_18245,m_1058,0,23,86400,94010,409.0,0.0,0.0,train
1,13432,c_23810,c_23810,m_13,0,23,86420,94520,310.0,0.0,0.0,train
2,95539,c_24416,c_24416,m_3160,0,23,86430,94800,372.0,0.0,0.0,train
3,109746,c_29090,c_29090,m_3470,0,23,86430,94880,722.0,0.0,0.0,train
4,59385,c_21483,c_21483,m_2296,0,23,86410,94930,278.0,0.0,0.0,train
5,77244,c_2344,c_2344,m_2683,0,23,86400,94940,558.0,0.0,0.0,train
6,49182,c_24860,c_24860,m_2115,0,23,86410,95240,518.0,0.0,0.0,train
7,69558,c_26410,c_26410,m_2520,0,23,86420,95310,387.0,0.0,0.0,train
8,1676,c_30032,c_30032,m_1027,0,23,86410,95340,58.0,0.0,0.0,train
9,143067,c_24550,c_24550,m_609,0,23,86400,95510,493.0,0.0,0.0,train


,windows
split,
train,114198
val,24471
test,24471


source                                          alibaba_raw_archives
num_windows                                                   163140
window_size                                                       24
num_features                                                       8
context_dim                                                       17
feature_columns    [cpu_util, mem_util, cpi, mem_gps, mpki, net_i...
context_columns    [container_cpu_request, container_cpu_limit, c...
Name: dataset_meta, dtype: object

## Save Dataset Artifacts

The saved arrays, metadata tables, and encoder artifacts are reused by the training, evaluation, and GPT adjudication notebooks. This keeps the Jupyter workflow reproducible while delegating the actual implementation to the modular Python package.


In [6]:
artifact_paths = save_dataset_bundle(bundle, config.paths.processed_dir)
display(pd.Series(artifact_paths, name="artifact_path"))


X                  C:\Users\kaspe\Desktop\Project\project\data\re...
C                  C:\Users\kaspe\Desktop\Project\project\data\re...
metadata           C:\Users\kaspe\Desktop\Project\project\data\re...
feature_meta       C:\Users\kaspe\Desktop\Project\project\data\re...
dataset_meta       C:\Users\kaspe\Desktop\Project\project\data\re...
context_encoder    C:\Users\kaspe\Desktop\Project\project\data\re...
build_summary      C:\Users\kaspe\Desktop\Project\project\data\re...
Name: artifact_path, dtype: str